# 🔍 Clinisys Silver Layer Reconciliation: Local DuckDB vs AWS Athena Production

This notebook automates the data quality reconciliation between the local **DuckDB** database (`clinisys_all.duckdb`, schema `silver`) and the production **AWS Athena** database (`silver_clinisys_prod`).

### Objectives:
1. **Row Count Audit**: Compare total records per table.
2. **Primary Key Overlap Audit**: Reconcile exact keys present in both environments, only locally, or only in production.
3. **Yearly Breakdown**: Drill down row counts and key matches per calendar year (using mapped date columns).
4. **Newest Record Analysis**: Fetch and show the most recent records present in only one environment (ordered by primary key descending).

### Database Connections:
- **Local**: DuckDB (`clinisys_all.duckdb` -> schema: `silver`)
- **AWS Athena**: PyAthena (`silver_clinisys_prod` database)

In [ ]:
import os
import yaml
import duckdb
import pandas as pd
import numpy as np
from pyathena import connect
import warnings
warnings.filterwarnings('ignore')

# Configuration
DUCKDB_PATH = '../../database/clinisys_all.duckdb'
ATHENA_REGION = 'sa-east-1'
ATHENA_WORKGROUP = 'datalake-admins'
ATHENA_DB = 'silver_clinisys_prod'
CONFIG_PATH = '../../clinisys/column_config.yml'

print("Libraries imported. Configured database paths:")
print(f"  Local DuckDB: {os.path.abspath(DUCKDB_PATH)}")
print(f"  AWS Athena:   {ATHENA_DB} (Region: {ATHENA_REGION}, Workgroup: {ATHENA_WORKGROUP})")

## 🔌 Connection Helpers
Defining robust wrapper functions to connect, execute queries, and guarantee connection closure.

In [ ]:
def run_duck(query):
    """Runs a query on local DuckDB, ensuring connection is closed."""
    conn = duckdb.connect(DUCKDB_PATH, read_only=True)
    try:
        return conn.execute(query).df()
    finally:
        conn.close()

def run_athena(query):
    """Runs a query on AWS Athena, ensuring connection is closed."""
    conn = connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB)
    try:
        return pd.read_sql(query, conn)
    finally:
        conn.close()

# Helper to get columns present in BOTH DuckDB and Athena schemas to prevent schema drift crashes
def get_common_columns(table):
    # 1. Local columns
    conn = duckdb.connect(DUCKDB_PATH, read_only=True)
    try:
        local_cols = set([c[0].lower() for c in conn.execute(f"SELECT * FROM silver.{table} LIMIT 0").description])
    finally:
        conn.close()
    
    # 2. Athena columns
    try:
        prod_df = run_athena(f"SELECT * FROM silver_clinisys_prod.{table} LIMIT 0")
        prod_cols = set([c.lower() for c in prod_df.columns])
    except Exception:
        prod_cols = set()
        
    return list(local_cols & prod_cols)

# Test connections
try:
    duck_ok = run_duck("SELECT 1 as test").iloc[0]['test'] == 1
    print("✅ DuckDB Connection: OK")
except Exception as e:
    print(f"❌ DuckDB Connection: FAILED - {e}")
    duck_ok = False

try:
    athena_ok = run_athena("SELECT 1 as test").iloc[0]['test'] == 1
    print("✅ AWS Athena Connection: OK")
except Exception as e:
    print(f"❌ AWS Athena Connection: FAILED - {e}")
    athena_ok = False

## 🗺️ Configuration & Predefined Mappings
Loading primary keys from `column_config.yml` and defining mappings of tables to their primary date columns for yearly breakdowns.

In [ ]:
# Load primary keys from configuration
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)
primary_keys = config.get('primary_keys', {})

# Map tables to their primary date column (determined from analysis)
TABLE_DATE_COLUMNS = {
    'view_agenda': 'data',
    'view_agendas': 'data',
    'view_congelamentos_embrioes': 'data_congelamento',
    'view_congelamentos_ovulos': 'data_congelamento',
    'view_congelamentos_semen': 'data_congelamento',
    'view_congelamentos_semen_doador': 'data_congelamento',
    'view_descongelamentos_embrioes': 'data_descongelamento',
    'view_descongelamentos_ovulos': 'data_descongelamento',
    'view_embrioes_congelados': 'data_congelamento',
    'view_exames': 'data',
    'view_extrato_atendimentos_central': 'data',
    'view_indicacao_novo': 'data',
    'view_medicamentos_prescricoes': 'data_inicial',
    'view_micromanipulacao': 'data',
    'view_micromanipulacao_oocitos': 'data_procedimento',
    'view_orcamentos': 'data_entrega_orcamento',
    'view_ovulos_congelados': 'data_congelamento',
    'view_procedimentos_financas': 'data_pagamento',
    'view_tratamentos': 'data_procedimento',
    'view_tratamentos_us_anexos': 'data'
}

print(f"Loaded {len(primary_keys)} primary key configs.")
print(f"Predefined date columns for {len(TABLE_DATE_COLUMNS)} tables.")

## 🔍 Identify Common Tables
Querying both databases to find matching tables in DuckDB `silver` schema and Athena `silver_clinisys_prod` database.

In [ ]:
# Query tables
local_tables_df = run_duck("SELECT table_name FROM information_schema.tables WHERE table_schema = 'silver'")
local_tables = sorted(local_tables_df['table_name'].tolist())

athena_tables_df = run_athena("SHOW TABLES")
athena_tables = sorted(athena_tables_df.iloc[:, 0].tolist())

common_tables = sorted(list(set(local_tables) & set(athena_tables)))
print(f"Found {len(local_tables)} tables in local DuckDB 'silver' schema.")
print(f"Found {len(athena_tables)} tables in Athena '{ATHENA_DB}' database.")
print(f"Common tables to compare ({len(common_tables)}):")
for t in common_tables:
    key_status = f"Key: {primary_keys.get(t)}" if t in primary_keys else "⚠️ No Primary Key in config"
    
    # Schema-aware date column checking
    try:
        cols = get_common_columns(t)
        date_col = TABLE_DATE_COLUMNS.get(t)
        if date_col and date_col.lower() not in cols:
            fallbacks = [c for c in cols if 'data' in c or 'date' in c]
            date_col = fallbacks[0] if fallbacks else None
    except Exception:
        date_col = None
        
    date_status = f"Date: {date_col}" if date_col else "No Date column (All Time)"
    print(f"  - {t:<35} | {key_status:<25} | {date_status}")

## 📊 Part 1: Row Count & Key Reconciliation Summary
Iterating through all common tables that have defined primary keys to perform count, match, and overlap checks.

In [ ]:
summary_data = []

for t in common_tables:
    if t not in primary_keys:
        continue
    
    key = primary_keys[t]
    
    # Row counts
    local_count = run_duck(f"SELECT COUNT(*) as cnt FROM silver.{t}").iloc[0]['cnt']
    prod_count = run_athena(f"SELECT COUNT(*) as cnt FROM silver_clinisys_prod.{t}").iloc[0]['cnt']
    
    # Fetch keys to calculate overlap
    local_keys_df = run_duck(f"SELECT {key} FROM silver.{t}")
    local_keys_df.columns = [c.lower() for c in local_keys_df.columns]
    local_keys = set(local_keys_df[key.lower()].dropna().tolist())
    
    prod_keys_df = run_athena(f"SELECT {key} FROM silver_clinisys_prod.{t}")
    prod_keys_df.columns = [c.lower() for c in prod_keys_df.columns]
    prod_keys = set(prod_keys_df[key.lower()].dropna().tolist())
    
    matched_keys = len(local_keys & prod_keys)
    only_local = len(local_keys - prod_keys)
    only_prod = len(prod_keys - local_keys)
    
    summary_data.append({
        'Table': t,
        'Primary Key': key,
        'Local Rows': local_count,
        'Athena Rows': prod_count,
        'Difference': local_count - prod_count,
        'Match Count': matched_keys,
        'Only in Local (DuckDB)': only_local,
        'Only in Prod (Athena)': only_prod
    })

summary_df = pd.DataFrame(summary_data)
summary_df.style.format({
    'Local Rows': '{:,}',
    'Athena Rows': '{:,}',
    'Difference': '{:+,}',
    'Match Count': '{:,}',
    'Only in Local (DuckDB)': '{:,}',
    'Only in Prod (Athena)': '{:,}'
}).bar(subset=['Difference'], align='mid', color=['#d65f5f', '#5fba7d'])

## 📅 Part 2: Yearly Breakdown Analysis
Drilling down row counts and key overlaps per year for each table. For tables without defined date columns, counts are grouped under 'N/A'.

In [ ]:
for t in common_tables:
    if t not in primary_keys:
        continue
        
    key = primary_keys[t]
    
    # Schema-aware date column extraction (common to both DBs)
    cols = get_common_columns(t)
    date_col = TABLE_DATE_COLUMNS.get(t)
    if date_col and date_col.lower() not in cols:
        fallbacks = [c for c in cols if 'data' in c or 'date' in c]
        date_col = fallbacks[0] if fallbacks else None
    
    print("=" * 80)
    print(f"📊 Table: {t} (Primary Key: {key} | Date Column: {date_col or 'N/A'})")
    print("=" * 80)
    
    # Fetch keys with date column
    if date_col:
        duck_q = f"SELECT {key} as key_val, {date_col} FROM silver.{t}"
        ath_q = f"SELECT {key} as key_val, {date_col} FROM silver_clinisys_prod.{t}"
        
        local_df = run_duck(duck_q)
        local_df.columns = [c.lower() for c in local_df.columns]
        
        prod_df = run_athena(ath_q)
        prod_df.columns = [c.lower() for c in prod_df.columns]
        
        # Normalize key and date column strings to lowercase
        key_lower = 'key_val'
        date_lower = date_col.lower()
        
        # Parse year safely using Pandas in Python to be 100% format-agnostic
        local_df['record_year'] = pd.to_datetime(local_df[date_lower], dayfirst=True, errors='coerce').dt.year
        local_df['record_year'] = local_df['record_year'].fillna('N/A').apply(lambda x: str(int(x)) if isinstance(x, (int, float)) and not pd.isna(x) else str(x))
        
        prod_df['record_year'] = pd.to_datetime(prod_df[date_lower], dayfirst=True, errors='coerce').dt.year
        prod_df['record_year'] = prod_df['record_year'].fillna('N/A').apply(lambda x: str(int(x)) if isinstance(x, (int, float)) and not pd.isna(x) else str(x))
    else:
        local_df = run_duck(f"SELECT {key} as key_val FROM silver.{t}")
        local_df.columns = [c.lower() for c in local_df.columns]
        local_df['record_year'] = 'N/A'
        
        prod_df = run_athena(f"SELECT {key} as key_val FROM silver_clinisys_prod.{t}")
        prod_df.columns = [c.lower() for c in prod_df.columns]
        prod_df['record_year'] = 'N/A'
        key_lower = 'key_val'
        
    # Unique years
    years = sorted(list(set(local_df['record_year'].dropna().tolist()) | set(prod_df['record_year'].dropna().tolist())))
    
    yearly_summary = []
    for yr in years:
        l_keys = set(local_df[local_df['record_year'] == yr][key_lower].dropna().tolist())
        p_keys = set(prod_df[prod_df['record_year'] == yr][key_lower].dropna().tolist())
        
        matched = len(l_keys & p_keys)
        only_l = len(l_keys - p_keys)
        only_p = len(p_keys - l_keys)
        
        yearly_summary.append({
            'Year': yr,
            'Local Count': len(l_keys),
            'Athena Count': len(p_keys),
            'Matched Count': matched,
            'Only Local': only_l,
            'Only Athena': only_p
        })
        
    yearly_df = pd.DataFrame(yearly_summary)
    display(yearly_df)
    print("\n")

## 🆕 Part 3: Mismatch Drill-Down — Newest Mismatched Records
Fetching and showcasing up to 5 newest records (ordered by primary key descending) that are exclusive to either the Local database or AWS Athena database.

In [ ]:
for t in common_tables:
    if t not in primary_keys:
        continue
        
    key = primary_keys[t]
    
    # Fetch keys from both
    local_keys_df = run_duck(f"SELECT {key} FROM silver.{t}")
    local_keys_df.columns = [c.lower() for c in local_keys_df.columns]
    local_keys = set(local_keys_df[key.lower()].dropna().tolist())
    
    prod_keys_df = run_athena(f"SELECT {key} FROM silver_clinisys_prod.{t}")
    prod_keys_df.columns = [c.lower() for c in prod_keys_df.columns]
    prod_keys = set(prod_keys_df[key.lower()].dropna().tolist())
    
    only_l_keys = list(local_keys - prod_keys)
    only_p_keys = list(prod_keys - local_keys)
    
    # Sort only keys descending
    only_l_keys.sort(reverse=True)
    only_p_keys.sort(reverse=True)
    
    print("=" * 80)
    print(f"🔍 Mismatch Samples: {t} (Primary Key: {key})")
    print("=" * 80)
    
    # Sample Local only
    if only_l_keys:
        sample_keys = only_l_keys[:]
        keys_placeholder = ", ".join([str(k) if isinstance(k, (int, float)) else f"'{k}'" for k in sample_keys])
        sample_q = f"SELECT * FROM silver.{t} WHERE {key} IN ({keys_placeholder}) ORDER BY {key} DESC"
        local_samples = run_duck(sample_q)
        print(f"🆕 Top {len(sample_keys)} NEWEST records ONLY found in Local DuckDB (Total: {len(only_l_keys)}):")
        display(local_samples)
    else:
        print("✅ No records found exclusively in Local DuckDB.")
        
    # Sample Athena only
    if only_p_keys:
        sample_keys = only_p_keys[:]
        keys_placeholder = ", ".join([str(k) if isinstance(k, (int, float)) else f"'{k}'" for k in sample_keys])
        sample_q = f"SELECT * FROM silver_clinisys_prod.{t} WHERE {key} IN ({keys_placeholder}) ORDER BY {key} DESC"
        prod_samples = run_athena(sample_q)
        print(f"🆕 Top {len(sample_keys)} NEWEST records ONLY found in Athena Production (Total: {len(only_p_keys)}):")
        display(prod_samples)
    else:
        print("✅ No records found exclusively in Athena Production.")
    print("\n")